# Synthetic NIAH Counting v5

One shared v2-style GPT-2 LM with learned absolute positional embeddings and a mixed thinking/non-thinking toggle.

## Colab Setup

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = ""  # Optional: set to your GitHub repo URL before running in a fresh Colab.
REPO_DIR = pathlib.Path('/content/Synthetic_NiaH_like_Count')
if 'google.colab' in sys.modules and REPO_URL and not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
print('cwd =', pathlib.Path.cwd())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])

## Runtime Settings

In [ ]:
PRESET = 'debug'  # 'debug' for artifact smoke test, 'main' for full run
STAGE = 'all'
DEVICE = 'cuda'  # change to 'cpu' if needed
OUT_ROOT = 'outputs/v5'
RUN_NAME = ''
ABLATE_NO_CONFLICT_MASK = False
TRACE_INDICES = False

args = [sys.executable, '-m', 'synthetic_niah_v5.run_v5', '--preset', PRESET, '--stage', STAGE, '--device', DEVICE, '--out-root', OUT_ROOT]
if RUN_NAME:
    args += ['--run-name', RUN_NAME]
if ABLATE_NO_CONFLICT_MASK:
    args += ['--ablate-no-conflict-mask']
if TRACE_INDICES:
    args += ['--trace-indices']
print(' '.join(args))

## Run Pipeline

In [ ]:
subprocess.check_call(args)
RUN_DIR = pathlib.Path(OUT_ROOT) / RUN_NAME if RUN_NAME else pathlib.Path(OUT_ROOT)
RUN_DIR

## Key Tables

In [ ]:
import pandas as pd

train_log = pd.read_csv(RUN_DIR / 'tables' / 'train_log.csv')
eval_by_step = pd.read_csv(RUN_DIR / 'tables' / 'eval_by_step.csv')
ambiguous = pd.read_csv(RUN_DIR / 'tables' / 'ambiguous_prefix.csv')
display(train_log.tail())
display(eval_by_step.tail(12))
display(ambiguous.tail())

## Key Figures

In [ ]:
from IPython.display import Image, display

for name in [
    'train_loss_by_step_and_mode.png',
    'final_accuracy_by_step_mode.png',
    'final_accuracy_by_count_mode.png',
    'trace_metrics_by_count.png',
    'ambiguous_prefix_probs_by_step.png',
]:
    path = RUN_DIR / 'figures' / name
    if path.exists():
        display(Image(filename=str(path)))

## Save to Google Drive

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = pathlib.Path('/content/drive/MyDrive/synthetic_niah_v5')
    drive_dir.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['bash', '-lc', f'cp -r {RUN_DIR} {drive_dir}/'])
    print('Saved to', drive_dir)

## Optional GitHub Save

In [ ]:
COMMIT_RESULTS = False
if COMMIT_RESULTS:
    subprocess.check_call(['git', 'status', '--short'])
    subprocess.check_call(['git', 'add', str(RUN_DIR)])
    subprocess.check_call(['git', 'commit', '-m', f'Add synthetic NIAH v5 {PRESET} results'])
    subprocess.check_call(['git', 'push'])